<a href="https://colab.research.google.com/github/goderdzi/goderdzi/blob/main/notebooks/piper_model_exporter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <font color="ffc800"> **[Piper](https://github.com/rhasspy/piper) model exporter.**
## ![Piper logo](https://contribute.rhasspy.org/img/logo.png)
---

* Notebook created by: [rmcpantoja](http://github.com/rmcpantoja)
* Collaborator: [Xx_Nessu_xX](http://github.com/XxNessuxX)

In [1]:
#@markdown # <font color="ffc800"> **Install software.** 📦
#@markdown ---

print("\033[93mInstalling...")
%cd /content
!git clone -q https://github.com/rmcpantoja/piper
%cd /content/piper/src/python
!pip install -q cython>=0.29.0 numpy>=1.26 librosa>=0.9.2 onnx onnxruntime-gpu pytorch_lightning
!bash build_monotonic_align.sh
!pip install -q --upgrade gdown
print("\033[93mDone!")

Installing...
/content
/content/piper/src/python
Compiling /content/piper/src/python/piper_train/vits/monotonic_align/core.pyx because it changed.
[1/1] Cythonizing /content/piper/src/python/piper_train/vits/monotonic_align/core.pyx
/usr/local/lib/python3.12/dist-packages/Cython/Compiler/Main.py:381: FutureWarning: Cython directive 'language_level' not set, using '3str' for now (Py3). This has changed from earlier releases! File: /content/piper/src/python/piper_train/vits/monotonic_align/core.pyx
  tree = Parsing.p_module(s, pxd, full_module_name)
performance hint: core.pyx:7:5: Exception check on 'maximum_path_each' will always require the GIL to be acquired.
Possible solutions:
	1. Declare 'maximum_path_each' as 'noexcept' if you control the definition and you're sure you don't want the function to raise exceptions.
	2. Use an 'int' return type on 'maximum_path_each' to allow an error code to be returned.
performance hint: core.pyx:38:6: Exception check on 'maximum_path_c' will alway

In [7]:
#@markdown # <font color="ffc800"> **Voice package generation section.** 🗣️
#@markdown ---
%cd /content/piper/src/python

import json
import ipywidgets as widgets
from IPython.display import display
from google.colab import output
guideurl = "https://github.com/rmcpantoja/piper/blob/master/notebooks/wav/en"
#@markdown #### *Download:*
#@markdown **Drive ID or direct download link of the model in another cloud:**
model_id = "https://drive.google.com/file/d/1Y0ehwDUKo_L2GrN5skbA1tbnJ7wdHyu3/view?usp=sharing" #@param {type:"string"}
#@markdown **Drive ID or direct download link of the config.json file:**
config_id = "https://drive.google.com/file/d/1rC-9ALvlN9wAD5enfRZZf4gwr6ym1l4D/view?usp=sharing" #@param {type:"string"}
#@markdown ---

#@markdown #### *Creation process:*
#@markdown **Choose the language code (iso639-1 format):**

#@markdown You can see a list of language codes and names [here](https://www.loc.gov/standards/iso639-2/php/English_list.php).

language = "en_US" #@param ["ar_JO", "ca_ES", "cs_CZ", "da_DK", "de_DE", "el_GR", "en_GB", "en_US", "es_ES", "es_LA", "fi_FI", "fr_FR", "grc", "hu_GU", "is_IS", "it_IT", "kk_KZ", "ka_GE", "lb_LU", "nb", "ne", "nl_BE", "no_NO", "pl_PL", "pt_BR", "pt_PT", "ro_RO", "ru_RU", "sk_SK", "sr", "sv_SE", "sw_CD", "tr_TR", "uk_UA", "vi_VN", "zh_CN"]
voice_name = "ucha" #@param {type:"string"}
voice_name = voice_name.lower()
quality = "medium" #@param ["high", "low", "medium", "x-low"]
#@markdown **Do you want to write a model card?** *(Optional.)*
write_model_card = True #@param {type:"boolean"}

#@markdown **Do you want this voice to have a faster response speed?**
streaming = True #@param {type:"boolean"}

def start_process(streaming):
    if not os.path.exists("/content/project/model.ckpt"):
        raise Exception("Could not download model! make sure the file is shareable to everyone")
    output.eval_js(f'new Audio("{guideurl}/starting.wav?raw=true").play()')
    if not streaming:
        !python -m piper_train.export_onnx "/content/project/model.ckpt" "{export_voice_path}/{export_voice_name}.onnx"
    else:
        !python -m piper_train.export_onnx_streaming "/content/project/model.ckpt" "{export_voice_path}"
    print("\033[93mCompressing...")
    !tar -czvf "{packages_path}/{export_voice_name}.tar.gz" -C "{export_voice_path}" .
    output.eval_js(f'new Audio("{guideurl}/success.wav?raw=true").play()')
    print("\033[93mDone!")

if not streaming:
    export_voice_name = f"{language}-{voice_name}-{quality}"
else:
    export_voice_name = f"{language}-{voice_name}+RT-{quality}"
export_voice_path = "/content/project/voice-"+export_voice_name
packages_path = "/content/project/packages"
if not os.path.exists(export_voice_path):
    os.makedirs(export_voice_path)
if not os.path.exists(packages_path):
    os.makedirs(packages_path)
print("\033[93mDownloading model and his config...")
if model_id.startswith("1"):
    !gdown -q "{model_id}" -O /content/project/model.ckpt
elif model_id.startswith("https://drive.google.com/file/d/"):
    !gdown -q "{model_id}" -O "/content/project/model.ckpt"
else:
    !wget "{model_id}" -O "/content/project/model.ckpt"
if config_id.startswith("1"):
    !gdown -q "{config_id}" -O "{export_voice_path}/{export_voice_name}.onnx.json"
elif config_id.startswith("https://drive.google.com/file/d/"):
    !gdown -q "{config_id}" -O "{export_voice_path}/{export_voice_name}.onnx.json"
else:
    !wget "{config_id}" -O "{export_voice_path}/{export_voice_name}.onnx.json"

if os.path.exists(f"{export_voice_path}/{export_voice_name}.onnx.json") and streaming:
    with open(f"{export_voice_path}/{export_voice_name}.onnx.json", "r", encoding="utf-8") as f:
        tmp = f.read()
    new_config = json.loads(tmp)
    new_config["streaming"] = True
    new_config["key"] = export_voice_name

    with open(f"{export_voice_path}/{export_voice_name}.onnx.json", "w", encoding="utf-8") as f_new:
        json.dump(new_config, f_new, indent=4)

if write_model_card:
    with open(f"{export_voice_path}/{export_voice_name}.onnx.json", "r") as file:
        config = json.load(file)
    sample_rate = config["audio"]["sample_rate"]
    num_speakers = config["num_speakers"]
    output.eval_js(f'new Audio("{guideurl}/waiting.wav?raw=true").play()')
    text_area = widgets.Textarea(
        description = "fill in this following template and press start to generate the voice package",
        value=f'# Model card for {voice_name} ({quality})\n\n* Language: {language} (normaliced)\n* Speakers: {num_speakers}\n* Quality: {quality}\n* Samplerate: {sample_rate}Hz\n\n## Dataset\n\n* URL: \n* License: \n\n## Training\n\nTrained from scratch.\nOr finetuned from: ',
        layout=widgets.Layout(width='500px', height='200px')
    )
    button = widgets.Button(description='Start')

    def create_model_card(button):
        model_card_text = text_area.value.strip()
        with open(f'{export_voice_path}/MODEL_CARD', 'w') as file:
            file.write(model_card_text)
        text_area.close()
        button.close()
        output.clear()
        start_process(streaming)

    button.on_click(create_model_card)

    display(text_area, button)
else:
    start_process(streaming)

/content/piper/src/python
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/piper/src/python/piper_train/export_onnx.py", line 12, in <module>
    from piper_train.vits.models import VitsModel
ImportError: cannot import name 'VitsModel' from 'piper_train.vits.models' (/content/piper/src/python/piper_train/vits/models.py)
Compressing...
./
./en_US--medium.onnx.json
Done!


In [6]:
%%writefile /content/piper/src/python/piper_train/export_onnx.py

import argparse
import logging
import onnxruntime as rt
import os
import sys
import time

import torch

# from piper_train.vits.models import SynthesizerTrn
from piper_train.vits.models import VitsModel

_LOGGER = logging.getLogger("export_onnx")


def main():
    parser = argparse.ArgumentParser(description="Export Piper model to ONNX")
    parser.add_argument("checkpoint", help="Path to checkpoint file (.ckpt)")
    parser.add_argument("output", help="Path to output ONNX file")
    parser.add_argument(
        "--config", type=str, default=None, help="Path to model config file (.json)"
    )

    args = parser.parse_args()
    _LOGGER.info(args)

    # Load model
    _LOGGER.info("Loading model from checkpoint: %s", args.checkpoint)
    # PATCH: Add weights_only=False to address _pickle.UnpicklingError in PyTorch 2.6+
    model = VitsModel.load_from_checkpoint(args.checkpoint, dataset=None, weights_only=False)
    model.eval()

    # Convert to ONNX
    _LOGGER.info("Converting to ONNX: %s", args.output)
    # Get example input
    # (text_lengths, text_input_ids)
    example_input = model.get_example_input()

    # Export ONNX
    torch.onnx.export(
        model,
        example_input,
        args.output,
        input_names=["text_lengths", "text_input_ids"],
        output_names=["phoneme_lengths", "phoneme_ids", "pitch_embeddings"],
        dynamic_axes={
            "text_lengths": {0: "batch_size"},
            "text_input_ids": {0: "batch_size", 1: "text_len"},
        },
        opset_version=17,
    )

    # Test ONNX model
    _LOGGER.info("Testing ONNX model")
    session_options = rt.SessionOptions()
    session_options.log_severity_level = 3  # ERROR

    onnx_session = rt.InferenceSession(args.output, session_options)

    _LOGGER.info("ONNX model tested successfully")

    if args.config:
        _LOGGER.info("Copying config to output directory")
        config_path = os.path.join(os.path.dirname(args.output), os.path.basename(args.config))
        import shutil
        shutil.copyfile(args.config, config_path)


if __name__ == "__main__":
    main()


Overwriting /content/piper/src/python/piper_train/export_onnx.py


In [8]:
#@markdown # <font color="ffc800"> **Download/export your generated voice package.** 📥
#@markdown ---

#@markdown #### *How do you want to export your model?*
export_mode = "Download the voice package on my device (may take some time)" #@param ["Download the voice package on my device (may take some time)", "upload it to my Google Drive"]
print("\033[93mExporting package...")
if export_mode == "Download the voice package on my device (may take some time)":
    from google.colab import files
    files.download(f"{packages_path}/{export_voice_name}.tar.gz")
    msg = "Please wait a moment while the package is being downloaded."
else:
    voicepacks_folder = "/content/drive/MyDrive/piper voice packages"
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists(voicepacks_folder):
        os.makedirs(voicepacks_folder)
    !cp "{packages_path}/{export_voice_name}.tar.gz" "{voicepacks_folder}"
    msg = f"You can find the generated voice package at: {voicepacks_folder}."
print(f"\033[93mDone! {msg}")

Exporting package...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done! Please wait a moment while the package is being downloaded.


# "*I want to test this model! I don't need anything else anymore?*"

No, this is almost the end! Now you can share your generated package to your friends, upload to a cloud storage and/or test it on:
* [The inference notebook](https://colab.research.google.com/github/rmcpantoja/piper/blob/master/notebooks/piper_inference_(ONNX).ipynb)
  * run the cells in order for it to work correctly, as well as all the notebooks. Also, the inference notebook will guide you through the process using the enhanced accessibility feature if you wish. It's easy to use. Test it!
* Or through the NVDA screen reader!
  * Download and install the latest version of the [add-on](https://github.com/mush42/piper-nvda/releases).
  * Once the add-on is installed, go to NVDA menu/piper voice manager...
  * In the installed voices page, tab until you find the `Install from local file` button, press enter and select the generated package in your downloads.
  * Once the package is selected and installed, apply the changes and restart NVDA to update the voice list.
* Enjoy your creation!